# AsistenteReg EP2
Martín Peña · ISY0101


In [ ]:
#@title 1 Cargar proyecto desde GitHub { display-mode: "form" }
import os
import sys
import subprocess
from pathlib import Path
from tempfile import mkdtemp

REPOSITORIO = "https://github.com/Martinignaciiooo/soluciones_ia_asistente_rag_v2.git"
COMMIT = "ef8f8ab572a389fac9dc03d5a9a1038cda788a1b"
RAIZ = Path(mkdtemp(prefix="asistentereg_github_", dir="/content"))

def ejecutar_git(*argumentos):
    try:
        return subprocess.run(
            ["git", "-C", str(RAIZ), *argumentos],
            check=True, capture_output=True, text=True, timeout=60,
            env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
        ).stdout.strip()
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError):
        raise RuntimeError("No se pudo descargar el proyecto de GitHub. Revisa la conexión y el acceso público al repositorio.") from None

ejecutar_git("init", "--quiet")
ejecutar_git("remote", "add", "origin", REPOSITORIO)
ejecutar_git("fetch", "--quiet", "--depth", "1", "origin", COMMIT)
ejecutar_git("checkout", "--quiet", "--detach", "FETCH_HEAD")
if ejecutar_git("rev-parse", "HEAD") != COMMIT:
    raise RuntimeError("La versión descargada no coincide con la versión fijada.")

necesarios = ("requirements.txt", "src/provider.py", "src/agent.py", "src/storage.py",
              "scripts/probar_base_datos.py", "scripts/exportar_evidencias.py")
if any(not (RAIZ / nombre).is_file() for nombre in necesarios):
    raise RuntimeError("El commit de GitHub no contiene la estructura completa del proyecto.")
proveedor = (RAIZ / "src/provider.py").read_text(encoding="utf-8")
almacenamiento = (RAIZ / "src/storage.py").read_text(encoding="utf-8")
agente = (RAIZ / "src/agent.py").read_text(encoding="utf-8")
if "OpenRouterEmbeddings" not in proveedor or "parallel_tool_calls=False" in proveedor.replace(" ", ""):
    raise RuntimeError("El proveedor del repositorio no corresponde a la versión corregida de OpenRouter.")
if any(texto not in almacenamiento for texto in ("ConversationBufferWindowMemory", "WINDOW_MESSAGES = 10", "DATABASE_URL")):
    raise RuntimeError("El repositorio no contiene la memoria PostgreSQL de diez mensajes.")
if "_answer_from_saved_draft" not in agente:
    raise RuntimeError("Falta la corrección de validación de citas del manual.")
if not list((RAIZ / "data/interna").glob("*.md")):
    raise RuntimeError("No se encontró el manual en data/interna del repositorio.")

for modulo in list(sys.modules):
    if modulo in ("src", "scripts") or modulo.startswith(("src.", "scripts.")):
        del sys.modules[modulo]
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
print("Proyecto descargado desde GitHub.")
print("Versión fijada:", COMMIT[:12])
print("Continúa con la instalación.")


## 2 Instalar y ejecutar pruebas sin usar la clave

In [ ]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], capture_output=True, text=True)
if r.returncode:
    print(r.stderr[-3000:])
    raise RuntimeError('Error al instalar dependencias')
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'], capture_output=True, text=True)
ruta_pruebas = RAIZ/'docs/evidencias/salida_pruebas_colab.txt'
ruta_pruebas.parent.mkdir(parents=True, exist_ok=True)
ruta_pruebas.write_text(r.stdout + r.stderr, encoding='utf-8')
print(r.stdout)
if r.returncode:
    raise RuntimeError('Hay pruebas fallidas; revisar antes de usar la API.')
print('Estas pruebas simulan la API y no consumen saldo.')


## 3 Configurar secretos y conectar PostgreSQL
En Secretos de Colab crea **OPENROUTER_API_KEY** y **DATABASE_URL**, y activa el acceso al cuaderno para ambos. DATABASE_URL es la cadena PostgreSQL de Neon o Supabase, sin el comando `psql` ni comillas. Las tablas del esquema `asistentereg` se crean automáticamente. Si faltan secretos se pedirá un campo oculto. Esta celda valida la base y los catálogos sin inferencia de IA.


In [ ]:
import getpass
from src.provider import OpenRouter, OpenRouterEmbeddings
from src.storage import Store
from src.retrieval import Retrieval
from src.agent import Agent

def cargar_secreto(nombre):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
    except Exception:
        valor = None
    if not valor:
        valor = os.environ.get(nombre)
    if not valor:
        valor = getpass.getpass(nombre + ': ')
    if not valor or not valor.strip():
        raise RuntimeError('Falta configurar ' + nombre)
    os.environ[nombre] = valor.strip()

for nombre in ('OPENROUTER_API_KEY', 'DATABASE_URL'):
    cargar_secreto(nombre)
os.environ['OPENROUTER_MODEL'] = 'openai/gpt-6-luna'
os.environ['OPENROUTER_EMBEDDING_MODEL'] = 'openai/text-embedding-3-small'
os.environ['LOCAL_BUDGET_USD'] = '0.75'
try:
    store = Store()
except Exception as error:
    raise RuntimeError('No se pudo conectar a PostgreSQL. Revisa DATABASE_URL y los permisos. Tipo: ' + type(error).__name__) from None
provider = OpenRouter(store)
print('PostgreSQL conectado; esquema asistentereg preparado. No se muestra la URL.')
print('GPT Luna configurado con effort low. No se enviaron solicitudes de inferencia.')
print('Consumo o reservas compartidos USD:', round(store.spent(), 6))


### Demostración de memoria en Neon sin consumir la API
Usa `ConversationBufferWindowMemory(k=5)` y PostgreSQL en Neon mediante `DATABASE_URL`. Conserva diez mensajes: cinco preguntas y cinco respuestas.

La prueba guarda seis intercambios en una conversación temporal, comprueba que permanezcan del segundo al sexto y elimina sus datos al terminar. No llama a OpenRouter.


In [ ]:
from scripts.probar_base_datos import verificar_base

if store.backend != 'postgresql':
    raise RuntimeError('Configura DATABASE_URL con la conexión de Neon antes de ejecutar esta demostración.')
resultado_base = verificar_base(store)
from scripts.exportar_evidencias import guardar_json
from datetime import datetime, timezone
resultado_base['fecha_ejecucion_utc'] = datetime.now(timezone.utc).isoformat()
guardar_json(RAIZ/'docs/evidencias/prueba_base_datos.json', resultado_base)
print('Almacenamiento: PostgreSQL en Neon')
print('Clase:', resultado_base['clase_memoria'], '| k:', resultado_base['k'])
print('Intercambios conservados:', resultado_base['intercambios_conservados'], '| Mensajes:', resultado_base['mensajes_conservados'])
print('Persistencia verificada:', resultado_base['persistencia_entre_conexiones'])
print('Intercambio más antiguo eliminado:', resultado_base['borrado_del_mas_antiguo'])
print('Llamadas a modelos:', resultado_base['llamadas_a_modelos'])
print('Datos de prueba eliminados:', resultado_base['datos_de_prueba_eliminados_al_final'])


## 4 Indexar el manual: chunking y embeddings
Se cargan los documentos originales de `data/interna`, con fragmentos de 700 caracteres y solapamiento de 120. Los embeddings usan OpenRouter y se reutilizan desde la caché de PostgreSQL.

**Esta etapa consume saldo sólo para textos que aún no tengan embeddings en caché.**


In [ ]:
retrieval = Retrieval(OpenRouterEmbeddings(provider), cache_dir=RAIZ/'runtime')
print('Documentos del manual indexados:')
for documento in retrieval.manifest['archivos']:
    print('-', documento['archivo'], '| Fragmentos:', documento['fragmentos'])
print('Chunking: 700 caracteres | Solapamiento: 120')
print('Fragmentos totales:', len(retrieval.fragments))
print('Modelo de embeddings:', retrieval.manifest['modelo_embeddings'])
print('Consumo o reservas acumuladas USD:', round(store.spent(), 6))


## 5 Demo real
Cada clic puede hacer llamadas pagadas de chat y embeddings. Máximo cuatro llamadas de chat por turno, con herramientas y memoria. Fecha simulada inicial: 6 de abril. Cambia al 11 de abril para mostrar el plazo vencido del retiro.

In [ ]:
import ipywidgets as widgets
from IPython.display import display,Markdown,clear_output
from datetime import date
import json
from scripts.exportar_evidencias import guardar_ejecucion
perfil=widgets.Text(value='grupo-ep2',description='Perfil:')
sesion=widgets.Text(value='demo-openrouter',description='Conversación:')
fecha=widgets.DatePicker(value=date(2026,4,6),description='Fecha simulada:')
consulta=widgets.Textarea(value='¿Hasta cuándo puedo retirar una asignatura? Prepara un borrador de consulta con los requisitos.',layout=widgets.Layout(width='95%',height='90px'))
boton=widgets.Button(description='Consultar',button_style='primary')
salida=widgets.Output()
def ejecutar(_):
    boton.disabled=True
    with salida:
        clear_output(wait=True)
        try:
            if not fecha.value:raise ValueError('Falta fecha')
            resultado=Agent(store,retrieval,provider,today=fecha.value.isoformat()).respond(consulta.value,sesion.value,perfil.value)
            display(Markdown(resultado['respuesta']))
            print(json.dumps(resultado['traza'],ensure_ascii=False,indent=2))
            for ident,titulo,cuerpo in resultado['borradores']:
                display(Markdown('### Borrador #'+str(ident)+' '+titulo+'\n\n'+cuerpo))
            evidencia = guardar_ejecucion(RAIZ, resultado, fecha.value.isoformat())
            print('Ensayo guardado:', evidencia.name)
            print('Consumo o reservas USD:',round(store.spent(),6))
        except Exception as error:
            print('No se pudo completar la consulta:',type(error).__name__)
            print('Revisa DATABASE_URL, conexión, clave, permisos de los dos modelos y saldo. No hubo reintento automático.')
    boton.disabled=False
boton.on_click(ejecutar)
display(perfil,sesion,fecha,consulta,boton,salida)


## 6 Historial persistente en PostgreSQL
La misma conversación y DATABASE_URL recuperan hasta 10 mensajes, incluso al reiniciar Colab o cambiar de equipo. Al completar el intercambio 6 se elimina el primero. Otra conversación empieza vacía. Los borradores, el consumo y la caché de embeddings se guardan también en la base remota. Reutiliza siempre la misma base del grupo para conservar el registro de consumo.


In [ ]:
for pregunta,respuesta in store.history(sesion.value):
    print('Estudiante:',pregunta)
    print('Asistente:',respuesta,'\n')

print("Historial, borradores, caché y consumo permanecen en PostgreSQL al cerrar Colab.")


## 7 Descargar evidencia
Descarga las pruebas, la verificación de Neon y todas las consultas ejecutadas en este entorno. Esta celda no llama a la API. Revisa los resultados y copia `docs/evidencias` al repositorio.


In [ ]:
from scripts.exportar_evidencias import exportar_evidencias
from google.colab import files
archivo_evidencias = exportar_evidencias(RAIZ)
files.download(str(archivo_evidencias))


## Información técnica
El chat usa GPT Luna con razonamiento `low` y los embeddings usan `text-embedding-3-small`, ambos por OpenRouter. No se ejecutan modelos locales ni se requiere GPU.

El agente conserva su estado en PostgreSQL mediante `ConversationBufferWindowMemory(k=5)`: diez mensajes, equivalentes a cinco preguntas y respuestas. Los intercambios anteriores se eliminan de esa conversación.

El corpus conserva los cuatro documentos de la primera entrega: reglamento académico, manual de titulación, calendario y preguntas frecuentes. El caso académico corresponde al semestre 2026-01 y la fecha de demo es simulada. Las pruebas y la comprobación de la base no consumen OpenRouter. La generación de vectores y las consultas reales sí consumen saldo.

La primera celda descarga el código y el manual desde GitHub. Usa el commit ef8f8ab572a389fac9dc03d5a9a1038cda788a1b para que la demo sea reproducible. No solicita subir archivos ni cargar el proyecto desde un ZIP. La descarga no llama a OpenRouter.

La entrega corregida omite `parallel_tool_calls` para mantener compatibilidad con GPT Luna y conserva `reasoning.effort=low`. Los errores HTTP incluyen un mensaje saneado del proveedor en la traza.

El cierre valida las citas de la respuesta. Si el modelo omite una cita pero existe un borrador del turno validado, reconstruye la respuesta desde las evidencias y el calendario comprobados; no agrega citas a afirmaciones inventadas ni hace llamadas adicionales. La traza distingue esta corrección.

La memoria de largo plazo recupera borradores guardados en Neon. Para probarla, usa la misma conversación y pregunta: «Recupera mi borrador anterior de retiro». Consulta hasta veinte borradores y devuelve hasta dos por similitud semántica. No recupera mensajes eliminados y puede consumir embeddings si faltan vectores en caché.

Repositorio de esta versión: [soluciones_ia_asistente_rag_v2](https://github.com/Martinignaciiooo/soluciones_ia_asistente_rag_v2). La versión del código queda fijada por COMMIT en la primera celda. La descarga es independiente del notebook almacenado en ese commit.


## Antes de entregar
Probar y revisar una consulta real, guardar evidencia, actualizar GitHub y ensayar la demo. Completar sin IA las conclusiones, justificaciones y reflexión personal del autor. Las pruebas preparadas sólo validan contratos simulados; la integración real requiere la clave del grupo.
